In [9]:
import pandas as pd

TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"]
metrics = ["r2_test", "mse_test", "r2_train", "mse_train"]
P = [1, 2, 3, 4, 5]

df_vs = pd.read_excel("Results-vs.xlsx", sheet_name=0)
df_no_vs = pd.read_excel("Results-no-vs.xlsx", sheet_name=0)

with pd.ExcelWriter("analise.xlsx", engine="openpyxl") as writer:
    for target in TARGETS:
        vs = df_vs[df_vs["target"] == target]
        no_vs = df_no_vs[df_no_vs["target"] == target]

        # Estatísticas
        stats_vs = vs[metrics].agg(["mean", "std"])
        stats_no_vs = no_vs[metrics].agg(["mean", "std"])

        # Top 10 por r2_test
        top_vs = vs.sort_values("r2_test", ascending=False).head(10)
        top_no_vs = no_vs.sort_values("r2_test", ascending=False).head(10)

        # Montar uma única sheet por target, em blocos
        sheet_name = target

        row = 0
        pd.DataFrame({"VS": ["Média", "Desvio Padrão"]}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )

        stats_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
        row += len(stats_vs) + 4

        pd.DataFrame({"NO-VS": ["Média", "Desvio Padrão"]}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )

        stats_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
        row += len(stats_no_vs) + 4

        pd.DataFrame({"VS - Top 10 r2_test": []}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )
        top_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)
        row += len(top_vs) + 4

        pd.DataFrame({"NO-VS - Top 10 r2_test": []}).to_excel(
            writer, sheet_name=sheet_name, startrow=row, index=False
        )
        top_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)


In [10]:
df_vs = pd.read_excel("Results-vs.xlsx", sheet_name=0)
df_no_vs = pd.read_excel("Results-no-vs.xlsx", sheet_name=0)

with pd.ExcelWriter("analise_2.xlsx", engine="openpyxl") as writer:
    for target in TARGETS:
        for p in sorted(df_vs["P"].dropna().unique()):
            vs = df_vs[(df_vs["target"] == target) & (df_vs["P"] == p)]
            no_vs = df_no_vs[(df_no_vs["target"] == target) & (df_no_vs["P"] == p)]

            if vs.empty and no_vs.empty:
                continue  # não cria aba vazia

            # Estatísticas
            stats_vs = vs[metrics].agg(["mean", "std"]) if not vs.empty else pd.DataFrame()
            stats_no_vs = no_vs[metrics].agg(["mean", "std"]) if not no_vs.empty else pd.DataFrame()

            # Top 10 por r2_test
            top_vs = vs.sort_values("r2_test", ascending=False).head(10) if not vs.empty else pd.DataFrame()
            top_no_vs = no_vs.sort_values("r2_test", ascending=False).head(10) if not no_vs.empty else pd.DataFrame()

            sheet_name = f"{target}_{int(p)}"
            row = 0

            # VS
            pd.DataFrame({"VS": ["Média", "Desvio Padrão"]}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not stats_vs.empty:
                stats_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
                row += len(stats_vs) + 4
            else:
                row += 4

            # NO-VS
            pd.DataFrame({"NO-VS": ["Média", "Desvio Padrão"]}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not stats_no_vs.empty:
                stats_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1)
                row += len(stats_no_vs) + 4
            else:
                row += 4

            # Top 10 VS
            pd.DataFrame({"VS - Top 10 r2_test": []}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not top_vs.empty:
                top_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)
                row += len(top_vs) + 4
            else:
                row += 4

            # Top 10 NO-VS
            pd.DataFrame({"NO-VS - Top 10 r2_test": []}).to_excel(
                writer, sheet_name=sheet_name, startrow=row, index=False
            )
            if not top_no_vs.empty:
                top_no_vs.to_excel(writer, sheet_name=sheet_name, startrow=row+1, index=False)


In [11]:
import pandas as pd

N = 10  # número de melhores modelos a considerar

df_vs = pd.read_excel("Results-vs.xlsx", sheet_name=0)
df_no_vs = pd.read_excel("Results-no-vs.xlsx", sheet_name=0)

# Unir as duas fontes (opcionalmente marque a origem)
df_vs["origem"] = "VS"
df_no_vs["origem"] = "NO-VS"
df_all = pd.concat([df_vs, df_no_vs], ignore_index=True)

sintese = []

for (target, p), g in df_all.groupby(["target", "P"]):
    top = g.sort_values("r2_test", ascending=False).head(N)

    ok = (top["r2_test"] > 0.5)

    qtd_ok = ok.sum()
    status = "Satisfatório" if qtd_ok > 0 else "Não satisfatório"

    # índice do melhor modelo
    idx_best = top["r2_test"].idxmax()

    sintese.append({
        "target": target,
        "P": int(p),
        "avaliados": len(top),
        "bons_modelos": int(qtd_ok),
        "status": status,
        "r2_test": top.loc[idx_best, "r2_test"],
        "mse_test": top.loc[idx_best, "mse_test"],
        "origem": top.loc[idx_best, "origem"],  # <<< aqui está a informação VS ou NO-VS
    })


df_sintese = pd.DataFrame(sintese).sort_values(["target", "P"])

print(df_sintese)
df_sintese.to_excel("sintese_modelos2.xlsx", index=False)

   target  P  avaliados  bons_modelos            status  r2_test     mse_test  \
0      Al  1         10             0  Não satisfatório  -1.8235    1163.8973   
1      Al  2         10             2      Satisfatório   0.6125    1379.6308   
2      Al  3         10             0  Não satisfatório  -3.5867    2648.4122   
3      Al  4         10            10      Satisfatório   0.6599     667.7760   
4      Al  5         10             0  Não satisfatório  -0.4079     928.2738   
5      As  1         10             3      Satisfatório   0.6865       0.0002   
6      As  2         10             5      Satisfatório   0.5412       0.0003   
7      As  3         10            10      Satisfatório   0.7797       0.0005   
8      As  4         10             0  Não satisfatório   0.4731       0.0010   
9      As  5         10             0  Não satisfatório   0.2736       0.0151   
10     Ba  1         10             1      Satisfatório   0.5040      27.9860   
11     Ba  2         10     